In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import os 
from thefuzz import process
import pandas as pd
import os
import re
import tkinter as tk
from tkinter import filedialog
from clase_reportes import ReportClass
from dateutil.relativedelta import relativedelta
import numpy as np
rc = ReportClass()


In [2]:
base_cuentas_clave = pd.read_excel(r"C:\Users\Dataa\Desktop\VENTAS\VENTA MENSUAL\data\cuentas_clave\base_cuentas_clave.xlsx")

# Inventarios


In [ ]:
# Importar inventarios
farmatodo = pd.read_html(r"C:\Users\Dataa\Desktop\VENTAS\VENTA MENSUAL\data\cuentas_clave\inventarios\DataStoreDetallado FARMATODO.xls")[0]
pasteur  = pd.read_excel(r"C:\Users\Dataa\Desktop\VENTAS\VENTA MENSUAL\data\cuentas_clave\inventarios\Inventario mes corriente_PASTEUR.xlsx")
locatel  = pd.read_excel(r"C:\Users\Dataa\Desktop\VENTAS\VENTA MENSUAL\data\cuentas_clave\inventarios\inventarioproveedor LOCATEL.xlsx")
prosalon = pd.read_csv(r"C:\Users\Dataa\Desktop\VENTAS\VENTA MENSUAL\data\cuentas_clave\inventarios\PROSALON.txt", delimiter='|', encoding='utf-8')
farmatodo = farmatodo.merge(base_cuentas_clave[['PRODUCTO','vpn_farmatodo']], left_on='VPN', right_on='vpn_farmatodo', how='left')
pasteur = pasteur.merge(base_cuentas_clave[['PRODUCTO','plu_pasteur']], left_on='PLU', right_on='plu_pasteur', how='left')
locatel = locatel.merge(base_cuentas_clave[['PRODUCTO','locatel_cod_sap']], left_on='CODIGO SAP', right_on='locatel_cod_sap', how='left')
prosalon = prosalon.merge(base_cuentas_clave[['PRODUCTO','item_prosalon']], left_on='ITEM', right_on='item_prosalon', how='left')
# Procesar locatel
cols = locatel.columns
# Identificamos las columnas tienda
columnas_fijas = ['EAN', 'CODIGO SAP', 'NOMBRE PRODUCTO', 'ESTADO',  'TOTAL', 'PRODUCTO','locatel_cod_sap']
# Columnas tiendas = todas las del medio
columnas_tiendas = [c for c in cols if c not in columnas_fijas ]

# Hacer el melt
locatel_pro = locatel.melt(
    id_vars=columnas_fijas,
    value_vars=columnas_tiendas,
    var_name='TIENDA',
    value_name='INVENTARIO'
)

# Crear columna adicional para identificar el proveedor
# Estandarizar dataframes
prosalon = prosalon[['PRODUCTO','Desc. C.O.', 'ITEM','DISPONIBLE']].reset_index(drop=True)
prosalon['CLIENTE'] = 'PROSALON'
prosalon = prosalon.rename(columns={'Desc. C.O.':'TIENDA', 'ITEM':'COD_CLIENTE', 'DISPONIBLE':'INVENTARIO'})


prosalon['TIENDA'] = prosalon['TIENDA'].str.replace(
    r'TIENDA|ARUMA', 
    '', 
    regex=True
).str.strip()


farmatodo = farmatodo[['PRODUCTO', 'Nombre de la tienda', 'VPN', 'Unidades disponibles en inventario','Máximo' ]].reset_index(drop=True)
farmatodo['CLIENTE'] = 'FARMATODO'
farmatodo = farmatodo.rename(columns={'Nombre de la tienda':'TIENDA', 'VPN':'COD_CLIENTE', 'Unidades disponibles en inventario':'INVENTARIO', 'Máximo':'MAXIMO'})
pasteur = pasteur[['PRODUCTO','PuntoVentaNombre','PLU','CantidadInventario' ]].reset_index(drop=True)
pasteur['CLIENTE'] = 'PASTEUR'
pasteur.rename(columns={'PuntoVentaNombre':'TIENDA', 'PLU':'COD_CLIENTE', 'CantidadInventario':'INVENTARIO'}, inplace=True)
locatel_pro = locatel_pro[['PRODUCTO','TIENDA','CODIGO SAP','INVENTARIO']].reset_index(drop=True)
locatel_pro['CLIENTE'] = 'LOCATEL'
locatel_pro.rename(columns={'Cantidad Disponible':'Unidades disponibles en inventario', 'CODIGO SAP':'COD_CLIENTE'}, inplace=True)

df_inventario = pd.concat([prosalon, farmatodo, pasteur, locatel_pro], ignore_index=True)

# Ventas

In [ ]:
ruta = r"C:\Users\Dataa\Desktop\VENTAS\VENTA MENSUAL\data\cuentas_clave"
carpetas = [f for f in os.listdir(ruta) if os.path.isdir(os.path.join(ruta, f))]
carpetas.remove('inventarios')

ventas_cuentas_clave = []

for i in carpetas:
    ruta_actual = os.path.join(ruta, i)

    if i.lower() == 'locatel':  
        df_consolidado = rc.consolidar_carpeta(
            ruta_carpeta=ruta_actual,
            extension='csv',
            encoding='latin-1',
            sep=';'
        )
        df_consolidado['CLIENTE'] = i.upper()

    elif i.lower() == 'prosalon':
        # Detectar archivos txt dentro de la carpeta
        archivos = [f for f in os.listdir(ruta_actual) if f.lower().endswith('.txt')]

        lista_dfs = []
        for archivo in archivos:
            ruta_archivo = os.path.join(ruta_actual, archivo)

            # Leer el txt separado por |
            df_txt = pd.read_csv(
                ruta_archivo,
                sep='|',
                engine='python',
                encoding='latin-1',
                
            )

            lista_dfs.append(df_txt)

        df_consolidado = pd.concat(lista_dfs, ignore_index=True)
        df_consolidado['CLIENTE'] = i.upper()

    else:
        df_consolidado = rc.consolidar_carpeta(ruta_carpeta=ruta_actual)
        df_consolidado['CLIENTE'] = i.upper()

    ventas_cuentas_clave.append(df_consolidado)


    

# Estandarizar las ventas
ventas_farmatodo = ventas_cuentas_clave[0].copy()
# Diccionario de meses en español
meses_es = {
    "enero": 1, "febrero": 2, "marzo": 3, "abril": 4,
    "mayo": 5, "junio": 6, "julio": 7, "agosto": 8,
    "septiembre": 9,  # por si viene así
    "octubre": 10, "noviembre": 11, "diciembre": 12
}

# Farmatodo

# Convertir a minúsculas para evitar errores
ventas_farmatodo["MES_NUM"] = ventas_farmatodo["MES"].str.strip().str.lower().map(meses_es)

# Crear columna FECHA (primer día del mes)
ventas_farmatodo["FECHA"] = pd.to_datetime(
    ventas_farmatodo["AÑO"].astype(str) + "-" + ventas_farmatodo["MES_NUM"].astype(str) + "-01"
)

ventas_farmatodo = ventas_farmatodo.merge(base_cuentas_clave[['PRODUCTO','vpn_farmatodo']], left_on='VPN', right_on='vpn_farmatodo', how='left')

ventas_farmatodo.rename(columns={'VPN':'COD_CLIENTE', 'TIENDA': 'N TIENDA','NOMBRE_TIENDA': 'TIENDA'}, inplace=True)
ventas_farmatodo = ventas_farmatodo[['CLIENTE','FECHA', 'TIENDA','COD_CLIENTE','PRODUCTO','CIUDAD', 'UNIDADES_VENDIDAS']]

# Locatel
ventas_locatel = ventas_cuentas_clave[1].copy()


# Convertir a minúsculas para evitar errores
ventas_locatel["MES_NUM"] = ventas_locatel["MES"].str.strip().str.lower().map(meses_es)
ventas_locatel["MES_NUM"] = ventas_locatel["MES_NUM"].astype(int)

# Crear columna FECHA (primer día del mes)
ventas_locatel["FECHA"] = pd.to_datetime(
    ventas_locatel["AÑO"].astype(str) + "-" + ventas_locatel["MES_NUM"].astype(str) + "-01"
)
ventas_locatel = ventas_locatel.merge(base_cuentas_clave[['PRODUCTO','locatel_cod_sap']], left_on='CODIGO SAP', right_on='locatel_cod_sap', how='left')

ventas_locatel.rename(columns={'NOMBRE CENTRO':'TIENDA', 'CODIGO SAP': 'COD_CLIENTE', 'UNIDADES VENDIDAS': 'UNIDADES_VENDIDAS'}, inplace=True)

ventas_locatel['CIUDAD'] = 'VARIAS'


ventas_locatel = ventas_locatel[['CLIENTE','FECHA','TIENDA','COD_CLIENTE','PRODUCTO', 'CIUDAD','UNIDADES_VENDIDAS']]


In [ ]:

# Novaventa
# ventas_novaventa = ventas_cuentas_clave[2].copy()
# fecha_campana = {1: '22-01', 2: '10-02', 3: '25-02', 4: '16-03',
#                  5: '3-04', 6: '24-04', 7: '15-05', 8: '3-06', 
#                  9: '24-06', 10: '14-07', 11: '31-07', 12: '22-08',
#                  13: '10-09', 14: '29-09', 15: '19-10', 16: '10-11',
#                  17: '02-12', 18: '22-12', 19: '31-12'
#                  }
# ventas_novaventa['FACHA_CAMPANA'] = ventas_novaventa['Campaña'].map(fecha_campana)
# ventas_novaventa['FECHA'] = pd.to_datetime(
#     ventas_novaventa['Año'].astype(str) + '-' + ventas_novaventa['FACHA_CAMPANA'].astype(str), format='%Y-%d-%m'
# )
# ventas_novaventa = ventas_novaventa.merge(base_cuentas_clave[['PRODUCTO','cod_novaventa']], left_on='Código', right_on='cod_novaventa', how='left')
# ventas_novaventa['TIENDA'] = 'NOVAVENTA'
# ventas_novaventa['CIUDAD'] = 'VARIAS'
# ventas_novaventa.rename(columns={'Código':'COD_CLIENTE', 'Unds Brutas':'UNIDADES_VENDIDAS'}, inplace=True)
# ventas_novaventa = ventas_novaventa[['CLIENTE','FECHA', 'TIENDA','COD_CLIENTE', 'PRODUCTO', 'CIUDAD', 'UNIDADES_VENDIDAS']]

# Pasteur
ventas_pasteur = ventas_cuentas_clave[3].copy()
ventas_pasteur = ventas_pasteur.merge(base_cuentas_clave[['PRODUCTO','plu_pasteur']], left_on='PLU', right_on='plu_pasteur', how='left')
ventas_pasteur.rename(columns={'PuntoVentaNombre':'TIENDA', 'PLU':'COD_CLIENTE', 
                               'CantidadVenta':'UNIDADES_VENDIDAS', 'Ciudad':'CIUDAD', 'fechaVenta':'FECHA'}, inplace=True)
ventas_pasteur = ventas_pasteur[['CLIENTE','FECHA', 'TIENDA','COD_CLIENTE', 'PRODUCTO', 'CIUDAD', 'UNIDADES_VENDIDAS']]
# df_ventas = pd.concat([ventas_farmatodo, ventas_locatel, ventas_pasteur], ignore_index=True)

# prosalon
ventas_prosalon = ventas_cuentas_clave[4].copy()
ventas_prosalon = ventas_prosalon.merge(base_cuentas_clave[['PRODUCTO','item_prosalon']], left_on='ITEM', right_on='item_prosalon', how='left')


ventas_prosalon.rename(columns={'Desc. C.O.':'TIENDA', 'ITEM':'COD_CLIENTE', 
                               'UNIDADES':'UNIDADES_VENDIDAS'}, inplace=True)
ventas_prosalon['CIUDAD'] = 'VARIAS'
ventas_prosalon = ventas_prosalon[['CLIENTE','FECHA', 'TIENDA','COD_CLIENTE', 'PRODUCTO', 'CIUDAD', 'UNIDADES_VENDIDAS']]

ventas_prosalon['FECHA'] = pd.to_datetime(
    ventas_prosalon['FECHA'].str.replace(r'\s+', ' ', regex=True).str.strip(),
   format="mixed"
)


ventas_prosalon['TIENDA'] = ventas_prosalon['TIENDA'].str.replace(
    r'TIENDA|ARUMA', 
    '', 
    regex=True
).str.strip()

df_ventas = pd.concat([ventas_farmatodo, ventas_locatel, ventas_prosalon, ventas_prosalon], ignore_index=True)

# PRONOSTICO


In [15]:
# --- 1. Preparación de Fechas ---
df_ventas['FECHA'] = pd.to_datetime(df_ventas['FECHA'])
df_ventas['FECHA_MES'] = df_ventas['FECHA'].dt.to_period('M').dt.to_timestamp()

# Definir periodo (últimos 3 meses cerrados)
ultima_fecha_global = df_ventas['FECHA_MES'].max()
inicio_mes_actual = ultima_fecha_global 
meses_analisis = 3
fecha_inicio = inicio_mes_actual - relativedelta(months=meses_analisis)

# Filtrar
df_historico = df_ventas[
    (df_ventas['FECHA_MES'] >= fecha_inicio) & 
    (df_ventas['FECHA_MES'] < inicio_mes_actual)
].copy()

# --- 2. Agrupación Mensual ---
# Sumamos ventas por mes específico
cols_agrupacion = ['CLIENTE', 'COD_CLIENTE', 'TIENDA', 'CIUDAD', 'PRODUCTO', 'FECHA_MES']
df_mensual = df_historico.groupby(cols_agrupacion)['UNIDADES_VENDIDAS'].sum().reset_index()

# --- 3. Asignación de Pesos (Lógica Temporal) ---
meses_ordenados = sorted(df_mensual['FECHA_MES'].unique())
# Mapa de pesos: Mes más antiguo = 1, Mes más reciente = 3
mapa_pesos = {fecha: i + 1 for i, fecha in enumerate(meses_ordenados)}

df_mensual['PESO'] = df_mensual['FECHA_MES'].map(mapa_pesos)

# Validar que tengamos los meses correctos (Debug)
print(f"Pesos asignados: {mapa_pesos}")

# --- 4. Cálculo de Venta Ponderada ---
df_mensual['VENTA_PONDERADA'] = df_mensual['UNIDADES_VENDIDAS'] * df_mensual['PESO']

# --- 5. Agrupación Final (Colapsar a nivel Producto) ---
# NOTA: Quitamos el 'mean' del agg porque es incorrecto para series temporales con ceros implícitos.
# Solo traemos las sumas.
df_rotacion = df_mensual.groupby(['CLIENTE', 'COD_CLIENTE', 'TIENDA', 'CIUDAD', 'PRODUCTO']).agg({
    'VENTA_PONDERADA': 'sum', 
    'UNIDADES_VENDIDAS': 'sum'
}).reset_index()

# --- 6. Cálculo de Métricas Finales ---

# A. MEDIA SIMPLE CORRECTA (Tu corrección)
# Dividimos la suma total por el número fijo de meses del análisis (3), no por los meses con venta.
df_rotacion['MEDIA_SIMPLE'] = df_rotacion['UNIDADES_VENDIDAS'] / meses_analisis

# B. MEDIA PONDERADA
suma_pesos_total = sum(mapa_pesos.values()) # 1+2+3 = 6
df_rotacion['MEDIA_PONDERADA'] = df_rotacion['VENTA_PONDERADA'] / suma_pesos_total

# --- 7. Lógica de Pronóstico (Smart Forecasting) ---
# Aquí definimos qué número creer dependiendo de la tendencia

# Definir umbral de sensibilidad (5% de diferencia)
umbral = 0.05

# Condiciones
cond_alcista = df_rotacion['MEDIA_PONDERADA'] > df_rotacion['MEDIA_SIMPLE'] * (1 + umbral)
cond_bajista = df_rotacion['MEDIA_PONDERADA'] < df_rotacion['MEDIA_SIMPLE'] * (1 - umbral)

# Lógica de Selección:
# 1. Si es ALCISTA: Usamos Ponderada (Para no quedarnos cortos de stock)
# 2. Si es BAJISTA: Usamos Ponderada (Para no comprar hueso, el mercado se está frenando)
# 3. Si es ESTABLE: Usamos Simple (Es más robusta y filtra el ruido)

condiciones = [cond_alcista, cond_bajista]
opciones_valor = [df_rotacion['MEDIA_PONDERADA'], df_rotacion['MEDIA_PONDERADA']]
opciones_etiqueta = ['ALCISTA', 'BAJISTA']

# Asignar valores
df_rotacion['PRONOSTICO_FINAL'] = np.select(condiciones, opciones_valor, default=df_rotacion['MEDIA_SIMPLE'])
df_rotacion['TENDENCIA'] = np.select(condiciones, opciones_etiqueta, default='ESTABLE')

# Redondeo para presentación (opcional)
df_rotacion['PRONOSTICO_FINAL'] = df_rotacion['PRONOSTICO_FINAL'].round(2)



Pesos asignados: {Timestamp('2025-09-01 00:00:00'): 1, Timestamp('2025-10-01 00:00:00'): 2, Timestamp('2025-11-01 00:00:00'): 3}


In [ ]:
DIAS_COBERTURA_OBJETIVO = {
    'FARMATODO': 8,
    'LOCATEL': 15,
    'PASTEUR': 15,
    'PROSALON': 15
}

In [17]:
# # Dependiendo del cliente
# DIAS_COBERTURA_OBJETIVO = 8 

# Excluir novaventa
# 1. Cruzar Inventario con el Pronóstico
# Usamos LEFT JOIN desde el inventario porque queremos ver todo lo que hay en bodega,
# incluso si no tiene pronóstico de venta (para detectar "huesos").
df_final = pd.merge(
    df_inventario, 
    df_rotacion, 
    on=['CLIENTE', 'COD_CLIENTE', 'TIENDA', 'PRODUCTO'], 
    how='left'
)


In [20]:

# 2. Limpieza de Nulos
df_final['PRONOSTICO_FINAL'] = df_final['PRONOSTICO_FINAL'].fillna(0)
df_final['TENDENCIA'] = df_final['TENDENCIA'].fillna('SIN MOVIMIENTO')

# 3. Transformación a Venta Diaria
# El pronóstico está mensual, lo pasamos a diario para aplicar los días de cobertura
df_final['VENTA_DIARIA_ESTIMADA'] = df_final['PRONOSTICO_FINAL'] / 30


In [21]:
# 4. Cálculo del Stock Ideal (Target)}
df_final['STOCK_OBJETIVO'] = df_final.apply(
    lambda row: row['VENTA_DIARIA_ESTIMADA'] * DIAS_COBERTURA_OBJETIVO.get(row['CLIENTE'], 15),
    axis=1 )

In [22]:
df_final['DIAS_COBERTURA_OBJETIVO'] = df_final.apply(
    lambda row: DIAS_COBERTURA_OBJETIVO.get(row['CLIENTE'], 15),
    axis=1 )

In [23]:
df_final['STOCK_OBJETIVO']  =  np.ceil(df_final['STOCK_OBJETIVO']).astype(int)

In [24]:
# df_final['STOCK_OBJETIVO'] = df_final['VENTA_DIARIA_ESTIMADA'] * DIAS_COBERTURA_OBJETIVO

# 5. Cálculo de Sugerencia de Reposición (Compra)
df_final['SUGERENCIA_COMPRA'] = df_final['STOCK_OBJETIVO'] - df_final['INVENTARIO']

# 6. Reglas de Negocio y Redondeo (La parte clave)
# A. Si el resultado es negativo (tengo de sobra), la sugerencia es 0 (no comprar negativo)
df_final['SUGERENCIA_COMPRA'] = df_final['SUGERENCIA_COMPRA'].clip(lower=0)

# B. Redondeo hacia ARRIBA (Ceiling)
df_final['SUGERENCIA_COMPRA'] = np.ceil(df_final['SUGERENCIA_COMPRA']).astype(int)



In [25]:
df_final[df_final.select_dtypes(include=['float64', 'int']).columns] = (
    df_final.select_dtypes(include=['float64', 'int']).fillna(0)
)


In [26]:
conditions = [
    (df_final['MAXIMO'] > 0) & (df_final['INVENTARIO'] == 0) & (df_final['CLIENTE'] == 'FARMATODO'),
    (df_final['MAXIMO'] == 0) & (df_final['UNIDADES_VENDIDAS'] > 0) & (df_final['CLIENTE'] == 'FARMATODO'),
    (df_final['MAXIMO'] == 0) & (df_final['INVENTARIO'] == 0) & (df_final['CLIENTE'] == 'FARMATODO'),
    (df_final['MAXIMO'] < df_final['STOCK_OBJETIVO']) & (df_final['CLIENTE'] == 'FARMATODO'),
    (df_final['INVENTARIO'] > 0) & (df_final['UNIDADES_VENDIDAS'] == 0),
    (df_final['MAXIMO'] > df_final['STOCK_OBJETIVO']),
    ((df_final['MAXIMO'] == df_final['STOCK_OBJETIVO']) & df_final['STOCK_OBJETIVO']> 0),
    (df_final['SUGERENCIA_COMPRA'] == 0),
    (df_final['UNIDADES_VENDIDAS']!= 0) & (df_final['INVENTARIO'] == 0),
    (df_final['SUGERENCIA_COMPRA']> 0)

]

choices = [
    'AGOTADO',
    'CON VENTAS SIN MAXIMO',
    'NO CODIFICADO',
    'AUMENTAR MAXIMO',
    'SIN VENTAS',
    'MAXIMO EXCEDIDO',
    'MAXIMO OPTIMO',
    'SIN COMPRA NECESARIA',
    'AGOTADO',
    'POCO STOCK'
]

df_final['CONDICIONAL'] = np.select(conditions, choices, default='REVISAR')



In [27]:
df_final['COBERTURA_%'] = np.where(
    df_final['UNIDADES_VENDIDAS'] != 0,
    df_final['INVENTARIO'] / df_final['STOCK_OBJETIVO'],
    1
)

In [28]:
df_final['COBERTURA_%'] = df_final['COBERTURA_%'].round(2)

In [29]:
df_final["RATIO_MAX_VS_STOCK_OBJETIVO"] = df_final["MAXIMO"] / df_final["STOCK_OBJETIVO"]


In [30]:
def clasificar(r):
    if r["INVENTARIO"] == 0 and r["MAXIMO"] < r["UNIDADES_VENDIDAS"]:
        return "AGOTADO + MAX < UNIDADES_VENDIDAS"
    if r["INVENTARIO"] == 0:
        return "AGOTADO"
    if r["MAXIMO"] < r["STOCK_OBJETIVO"]:
        return "MAX < STOCK_OBJETIVO (CRITICO)"
    if r["RATIO_MAX_VS_STOCK_OBJETIVO"] <= 1.2 and r["INVENTARIO"] <= 0.3 * r["MAXIMO"]:
        return "POCO MARGEN + ALTO RIESGO"
    if r["RATIO_MAX_VS_STOCK_OBJETIVO"] <= 1.2:
        return "MAX AJUSTADO"
    if r["INVENTARIO"] <= 0.2 * r["MAXIMO"]:
        return "INVENTARIO BAJO"
    return "OK"

df_final["RIESGO"] = df_final.apply(clasificar, axis=1)

In [31]:
df_final.to_excel(r"C:\Users\Dataa\Desktop\reporte_reposicion_cuentas_clave.xlsx", index=False)

In [32]:
# Exportar en varias hojas
ruta = r"C:\Users\Dataa\Desktop\reporte_reposicion_cuentas_clave2.xlsx"
clientes = df_final['CLIENTE'].unique().tolist()
with pd.ExcelWriter(ruta, engine='openpyxl') as writer:
    for cliente in clientes:
        df = df_final[df_final['CLIENTE'] == cliente]
        df.to_excel(writer, sheet_name=str(cliente), index=False)
